# BYV-AI — Test Set Evaluation v0.1

Notebook ini menjalankan **inference** adapter BYV-AI yang sudah dilatih pada **35 contoh test terpisah**. Notebook ini tidak melakukan training ulang.

**Sebelum mulai:**
1. Buka notebook ini di Google Colab dan pilih GPU T4 (atau GPU yang kompatibel).
2. Pastikan adapter hasil training masih ada di Google Drive pada `MyDrive/BYV-AI/outputs/byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2/`.
3. Notebook akan otomatis mengunduh `data/test.jsonl` dari repository GitHub ke `MyDrive/BYV-AI/data/test.jsonl`. Jika unduhan gagal, lu bisa upload file secara manual.
4. Jalankan cell dari atas ke bawah.

Output disimpan di `MyDrive/BYV-AI/experiments/byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2/test_evaluation/`:
- `test_predictions.jsonl` — prompt, target referensi, dan prediksi model.
- `test_results.csv` — tabel untuk review per contoh.
- `test_summary.json` — ringkasan pengecekan otomatis.

Pengecekan otomatis **bukan penilaian semantik**. Model bisa menghasilkan jawaban yang formatnya valid tetapi isinya salah. Baca semua prediksi dan tandai kualitas faktual, kelengkapan kode, dan ketepatan DSML secara manual sebelum menyimpulkan kualitas model.

In [1]:
# Instal stack inference dengan versi yang sama seperti training run BYV-AI v0.1.2.
# Bila Colab meminta restart runtime setelah instalasi, restart lalu jalankan notebook dari awal.
%pip -q install "transformers==5.22.1" "accelerate==1.15.0" "peft==0.21.2" "bitsandbytes==0.50.2"

from pathlib import Path
import json, re, time, csv, collections, os

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("PyTorch:", torch.__version__)
print("Transformers:", __import__('transformers').__version__)
print("PEFT:", __import__('peft').__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_ID = "Qwen/Qwen3-1.7B"
RUN_NAME = "byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2"
DRIVE_ROOT = Path('/content/drive/MyDrive/BYV-AI')
ADAPTER_DIR = DRIVE_ROOT / 'outputs' / RUN_NAME
DATA_DIR = DRIVE_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
TEST_PATH = DATA_DIR / 'test.jsonl'
RESULT_DIR = DRIVE_ROOT / 'experiments' / RUN_NAME / 'test_evaluation'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA tidak terdeteksi. Pilih Runtime > Change runtime type > T4 GPU.")
if not (ADAPTER_DIR / 'adapter_config.json').exists():
    raise FileNotFoundError(
        f"Adapter tidak ditemukan di {ADAPTER_DIR}. "
        "Pastikan output hasil training tersimpan di Google Drive."
    )
print("Adapter:", ADAPTER_DIR)
print("Test data:", TEST_PATH)
print("Output:", RESULT_DIR)

Mounted at /content/drive
Adapter: /content/drive/MyDrive/BYV-AI/outputs/byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2
Test data: /content/drive/MyDrive/BYV-AI/data/test.jsonl
Output: /content/drive/MyDrive/BYV-AI/experiments/byv-ai-qwen3-1.7b-qlora-t4-fp16-v0.1.2/test_evaluation


In [3]:
# Unduh selalu versi resmi terbaru dari test split di GitHub agar Drive tidak memakai file test lama.
TEST_URL = "https://raw.githubusercontent.com/naufalelghani/byv-ai/main/data/test.jsonl"
downloaded = False
try:
    from urllib.request import urlopen
    with urlopen(TEST_URL, timeout=30) as response:
        test_bytes = response.read()
    downloaded_lines = [line for line in test_bytes.decode('utf-8').splitlines() if line.strip()]
    if len(downloaded_lines) != 35:
        raise ValueError(f"Ekspektasi 35 baris test set, ditemukan {len(downloaded_lines)}.")
    parsed_download = []
    for line_no, line in enumerate(downloaded_lines, 1):
        try:
            parsed_download.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"JSONL GitHub tidak valid di baris {line_no}: {exc}") from exc
    downloaded_ids = [row.get('id') for row in parsed_download]
    if any(not item for item in downloaded_ids) or len(set(downloaded_ids)) != 35:
        raise ValueError("Test set GitHub memiliki ID kosong atau duplikat.")
    TEST_PATH.write_bytes(test_bytes)
    downloaded = True
    print("Test set v0.1 berhasil diperbarui dari GitHub:", TEST_PATH)
except Exception as download_error:
    print("Unduhan otomatis dari GitHub gagal:", repr(download_error))
    if TEST_PATH.exists():
        print("Menggunakan test.jsonl yang sudah ada di Google Drive. Pastikan itu versi v0.1 yang benar.")
    else:
        from google.colab import files
        print("Upload data/test.jsonl dari repository sebagai fallback.")
        uploaded = files.upload()
        candidate = uploaded.get('test.jsonl') or uploaded.get('BYV-AI_test_v0.1.jsonl')
        if candidate is None:
            raise FileNotFoundError("Upload test.jsonl atau BYV-AI_test_v0.1.jsonl untuk melanjutkan.")
        TEST_PATH.write_bytes(candidate)
        print("Test set tersimpan:", TEST_PATH)

print("Sumber test set:", "GitHub" if downloaded else "Google Drive / upload fallback")

Test set berhasil diunduh dari GitHub: /content/drive/MyDrive/BYV-AI/data/test.jsonl


In [7]:
# Load dan validasi test set. Target jawaban adalah pesan assistant paling akhir.
DSML_OPEN = "<｜DSML｜ calls>"
DSML_CLOSE = "</｜DSML｜ calls>"
VALID_ROLES = {"system", "user", "assistant"}

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL error baris {line_no}: {exc}")
    return rows

test_records = load_jsonl(TEST_PATH)
if len(test_records) != 35:
    raise ValueError(f"Ekspektasi test set v0.1 = 35 contoh, tetapi ditemukan {len(test_records)}.")

seen_ids = set()
for i, row in enumerate(test_records, 1):
    if not isinstance(row.get('messages'), list) or len(row['messages']) < 2:
        raise ValueError(f"Record {i} tidak memiliki messages yang valid.")
    if row['messages'][-1].get('role') != 'assistant':
        raise ValueError(f"Record {i}: pesan terakhir harus assistant sebagai target referensi.")
    if any(m.get('role') not in VALID_ROLES for m in row['messages']):
        raise ValueError(f"Record {i}: role tidak valid.")
    rid = row.get('id')
    if rid in seen_ids:
        raise ValueError(f"ID duplikat pada test set: {rid}")
    seen_ids.add(rid)

# Cek tidak ada ID yang overlap dengan train/eval bila file-file itu tersedia.
for split_name in ('train', 'eval'):
    split_path = DATA_DIR / f'{split_name}.jsonl'
    if split_path.exists():
        split_ids = {r.get('id') for r in load_jsonl(split_path)}
        overlap = seen_ids & split_ids
        if overlap:
            raise ValueError(f"DATA LEAKAGE: {len(overlap)} ID test juga ada di {split_name}: {sorted(overlap)[:5]}")
        print(f"ID test vs {split_name}: tidak overlap")
    else:
        print(f"Catatan: {split_path} tidak ditemukan; overlap dengan {split_name} tidak dapat diverifikasi.")

category_counts = collections.Counter(r.get('category', 'uncategorized') for r in test_records)
print("Jumlah contoh:", len(test_records))
print("Distribusi kategori:", dict(category_counts))

ID test vs train: tidak overlap
ID test vs eval: tidak overlap
Jumlah contoh: 35
Distribusi kategori: {'multi_turn_sequence': 2, 'clarification_safety': 2, 'dsml_single_tool': 4, 'reasoning': 3, 'markdown': 3, 'plain_text_with_tools': 3, 'code_javascript_typescript': 3, 'plain_text': 3, 'code_python': 3, 'code_html_css': 3, 'code_sql_other': 2, 'dsml_multiple_independent': 2, 'tool_result_to_markdown': 2}


In [8]:
# Load base model 4-bit NF4 + adapter LoRA. Pada T4 (compute capability 7.5) gunakan FP16.
capability = torch.cuda.get_device_capability(0)
COMPUTE_DTYPE = torch.bfloat16 if capability[0] >= 8 else torch.float16
print("GPU compute capability:", f"{capability[0]}.{capability[1]}")
print("Compute dtype:", COMPUTE_DTYPE)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer_source = str(ADAPTER_DIR) if (ADAPTER_DIR / 'tokenizer_config.json').exists() else MODEL_ID
TOKENIZER = AutoTokenizer.from_pretrained(tokenizer_source, use_fast=True)
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token

import transformers
major_version = int(transformers.__version__.split('.')[0])
dtype_arg = {'dtype': COMPUTE_DTYPE} if major_version >= 5 else {'torch_dtype': COMPUTE_DTYPE}
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    **dtype_arg,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False)
model.eval()
print("Model + adapter berhasil dimuat.")

GPU compute capability: 7.5
Compute dtype: torch.float16


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
# Validator DSML struktural (tidak mengeksekusi tool).
def validate_dsml(text):
    errors = []
    stripped = text.strip()
    if stripped.count(DSML_OPEN) != 1:
        errors.append("harus tepat satu outer calls opener")
    if stripped.count(DSML_CLOSE) != 1:
        errors.append("harus tepat satu outer calls closer")
    if not stripped.startswith(DSML_OPEN) or not stripped.endswith(DSML_CLOSE):
        errors.append("prose di luar calls block atau wrapper tidak lengkap")

    invokes = re.findall(r'<｜DSML｜ invoke name="([^"]+)">', stripped)
    if not invokes:
        errors.append("tidak ada invoke")
    if len(invokes) != stripped.count("</｜DSML｜ invoke>"):
        errors.append("jumlah invoke opener/closer tidak cocok")

    params = re.findall(r'<｜DSML｜ parameter name="([^"]+)" string="(true|false)">', stripped)
    if len(params) != stripped.count("</｜DSML｜ parameter>"):
        errors.append("jumlah parameter opener/closer tidak cocok")

    pattern = r'<｜DSML｜ parameter name="([^"]+)" string="(true|false)">(.*?)</｜DSML｜ parameter>'
    for name, is_string, value in re.findall(pattern, stripped, flags=re.DOTALL):
        if is_string == 'false':
            try:
                json.loads(value)
            except json.JSONDecodeError:
                errors.append(f"parameter non-string {name} bukan JSON valid")
    return {"valid": not errors, "errors": errors, "invoke_count": len(invokes), "invoke_names": invokes}

def extract_final(raw):
    # Qwen3 dapat mengeluarkan blok thinking sebelum </think>.
    if "</think>" in raw:
        final = raw.rsplit("</think>", 1)[1]
    elif "<think>" in raw:
        # Thinking tidak selesai; jangan menyamarkannya sebagai jawaban final.
        return "", True
    else:
        final = raw
    for token in ("<|im_end|>", "<|endoftext|>"):
        final = final.split(token, 1)[0]
    return final.strip(), False

def generate_one(messages, max_new_tokens=1200):
    prompt = TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    input_device = model.get_input_embeddings().weight.device
    inputs = TOKENIZER(prompt, return_tensors='pt').to(input_device)
    started = time.time()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # deterministic untuk evaluasi yang dapat diulang
            pad_token_id=TOKENIZER.pad_token_id,
            eos_token_id=(model.generation_config.eos_token_id or TOKENIZER.eos_token_id),
            use_cache=True,
        )
    elapsed = time.time() - started
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    raw = TOKENIZER.decode(new_tokens, skip_special_tokens=False)
    final, unfinished_thinking = extract_final(raw)
    return {"raw": raw, "prediction": final, "unfinished_thinking": unfinished_thinking, "latency_seconds": round(elapsed, 3)}

print("Validator dan fungsi inference siap.")

In [ ]:
# Jalankan 35 contoh. Untuk multi-turn, semua pesan sebelum assistant target terakhir dipertahankan.
results = []
for index, record in enumerate(test_records, 1):
    conversation = record['messages']
    prompt_messages = conversation[:-1]
    reference = conversation[-1]['content']

    generated = generate_one(prompt_messages)
    prediction = generated['prediction']
    reference_has_dsml = DSML_OPEN in reference
    prediction_has_dsml = DSML_OPEN in prediction
    dsml_check = validate_dsml(prediction) if prediction_has_dsml else None

    row = {
        "id": record.get('id'),
        "category": record.get('category', 'uncategorized'),
        "input_messages": prompt_messages,
        "reference": reference,
        "prediction": prediction,
        "raw_generation": generated['raw'],
        "reference_has_dsml": reference_has_dsml,
        "prediction_has_dsml": prediction_has_dsml,
        "dsml_structure_valid": dsml_check['valid'] if dsml_check else None,
        "dsml_errors": dsml_check['errors'] if dsml_check else [],
        "dsml_behavior_match": prediction_has_dsml == reference_has_dsml,
        "prediction_nonempty": bool(prediction.strip()),
        "unfinished_thinking": generated['unfinished_thinking'],
        "latency_seconds": generated['latency_seconds'],
    }
    results.append(row)
    print(f"[{index:02d}/{len(test_records)}] {row['id']} | {row['category']} | {row['latency_seconds']}s | DSML={prediction_has_dsml}")

print("Inference selesai:", len(results), "contoh")

In [ ]:
# Ringkasan pemeriksaan otomatis dan ekspor hasil.
dsml_expected = [r for r in results if r['reference_has_dsml']]
dsml_generated = [r for r in results if r['prediction_has_dsml']]
summary = {
    "run_name": RUN_NAME,
    "base_model": MODEL_ID,
    "adapter_dir": str(ADAPTER_DIR),
    "test_path": str(TEST_PATH),
    "test_examples": len(results),
    "category_counts": dict(collections.Counter(r['category'] for r in results)),
    "nonempty_predictions": sum(r['prediction_nonempty'] for r in results),
    "unfinished_thinking_count": sum(r['unfinished_thinking'] for r in results),
    "reference_dsml_count": len(dsml_expected),
    "prediction_dsml_count": len(dsml_generated),
    "dsml_behavior_matches": sum(r['dsml_behavior_match'] for r in results),
    "dsml_structurally_valid": sum(bool(r['dsml_structure_valid']) for r in dsml_generated),
    "dsml_structurally_invalid": sum(not bool(r['dsml_structure_valid']) for r in dsml_generated),
    "mean_latency_seconds": round(sum(r['latency_seconds'] for r in results) / max(1, len(results)), 3),
    "warning": "Automated checks are structural only; semantic correctness requires human review. No DSML tool is executed by this notebook."
}

pred_path = RESULT_DIR / 'test_predictions.jsonl'
with open(pred_path, 'w', encoding='utf-8') as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

csv_path = RESULT_DIR / 'test_results.csv'
fieldnames = [
    'id','category','reference_has_dsml','prediction_has_dsml','dsml_structure_valid',
    'dsml_behavior_match','prediction_nonempty','unfinished_thinking','latency_seconds',
    'reference','prediction','dsml_errors'
]
with open(csv_path, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    for row in results:
        writer.writerow(row)

summary_path = RESULT_DIR / 'test_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSaved:", pred_path)
print("Saved:", csv_path)
print("Saved:", summary_path)

## Cara membaca hasil

- `dsml_behavior_match`: apakah model mengeluarkan DSML pada contoh yang target referensinya DSML, dan tidak mengeluarkan DSML pada contoh yang targetnya bukan DSML. Ini indikator perilaku format, bukan bukti tool benar.
- `dsml_structure_valid`: memeriksa wrapper, jumlah tag, dan JSON pada parameter `string="false"`. Ini **tidak** memvalidasi nama tool/parameter terhadap schema dinamis.
- `prediction_nonempty`: memastikan ada jawaban final yang diekstrak setelah `</think>`.
- `unfinished_thinking`: model mencapai batas token sebelum menyelesaikan blok thinking/final.

**Jangan menjalankan DSML di notebook ini.** DSML hanya diperiksa sebagai teks. Gateway terpisah harus tetap memvalidasi schema, permission, path, dan command sebelum mengeksekusi tool.

Setelah selesai, download atau upload kembali `test_predictions.jsonl`, `test_results.csv`, dan `test_summary.json` untuk direview. Review manual minimal mencakup: ketepatan isi, kualitas reasoning ringkas, syntax/kelengkapan kode, Markdown, tool selection, dan kesesuaian nama parameter DSML dengan system prompt di setiap contoh.